In [0]:
%sql
-- =====================================================
-- CREATE DIMENSION TABLE: dim_product (Enhanced)
-- =====================================================
-- Source: silver_catalog.silver_schema.products
--         silver_catalog.silver_schema.invoice_line_items
-- Target: gold_catalog.gold_schema.dim_product
-- Purpose: Product dimension with surrogate key and cost_price imputation
-- =====================================================

CREATE OR REPLACE TABLE gold_catalog.gold_schema.dim_product AS
WITH avg_prices AS (
    SELECT
        product_id,
        AVG(unit_price) AS avg_sell_price
    FROM silver_catalog.silver_schema.invoice_line_items
    GROUP BY product_id
),

category_avg AS (
    SELECT
        p.category,
        AVG(li.unit_price) AS category_avg_price
    FROM silver_catalog.silver_schema.products p
    JOIN silver_catalog.silver_schema.invoice_line_items li
        ON p.product_id = li.product_id
    GROUP BY p.category
)

SELECT
    monotonically_increasing_id() AS product_key,
    p.product_id,
    p.product_name,
    p.category,

    ROUND(
        COALESCE(
            p.cost_price,
            ap.avg_sell_price * 0.7,
            ca.category_avg_price * 0.7
        ), 2
    ) AS cost_price

FROM silver_catalog.silver_schema.products p
LEFT JOIN avg_prices ap
    ON p.product_id = ap.product_id
LEFT JOIN category_avg ca
    ON p.category = ca.category
ORDER BY p.product_id;

In [0]:
%sql
select * from gold_catalog.gold_schema.dim_product;

In [0]:
%sql
-- =====================================================
-- CREATE DIMENSION TABLE: dim_customer
-- =====================================================
-- Source: silver_catalog.silver_schema.customers
-- Target: gold_catalog.gold_schema.dim_customer
-- Purpose: Customer dimension with surrogate key
-- =====================================================

CREATE OR REPLACE TABLE gold_catalog.gold_schema.dim_customer AS
SELECT
    monotonically_increasing_id() AS customer_key,
    customer_id,
    customer_name,
    customer_type,
    country,
    signup_date
FROM silver_catalog.silver_schema.customers
ORDER BY customer_id;

In [0]:
%sql
-- =====================================================
-- CREATE DIMENSION TABLE: dim_region
-- =====================================================
-- Source: silver_catalog.silver_schema.regions
-- Target: gold_catalog.gold_schema.dim_region
-- Purpose: Region dimension with surrogate key
-- =====================================================

CREATE OR REPLACE TABLE gold_catalog.gold_schema.dim_region AS
SELECT
    monotonically_increasing_id() AS region_key,
    region_code,
    region_name,
    country
FROM silver_catalog.silver_schema.regions
ORDER BY region_code;

In [0]:
%sql
-- =====================================================
-- CREATE DIMENSION TABLE: dim_date
-- =====================================================
-- Source: silver_catalog.silver_schema.invoices
-- Target: gold_catalog.gold_schema.dim_date
-- Purpose: Date dimension with surrogate key and date attributes
-- =====================================================

CREATE OR REPLACE TABLE gold_catalog.gold_schema.dim_date AS
SELECT
    monotonically_increasing_id() AS date_key,
    invoice_date AS full_date,
    YEAR(invoice_date) AS year,
    MONTH(invoice_date) AS month,
    DAY(invoice_date) AS day,
    DATE_FORMAT(invoice_date, 'EEEE') AS weekday
FROM (
    SELECT DISTINCT invoice_date
    FROM silver_catalog.silver_schema.invoices
)
ORDER BY invoice_date;

In [0]:
%sql
-- =====================================================
-- CREATE FACT TABLE: fact_sales
-- =====================================================
-- Sources: silver_catalog.silver_schema.invoice_line_items
--          silver_catalog.silver_schema.invoices
--          silver_catalog.silver_schema.exchange_rates
--          gold_catalog.gold_schema.dim_* (all dimensions)
-- Target:  gold_catalog.gold_schema.fact_sales
-- Purpose: Fact table with sales transactions and metrics
-- =====================================================

CREATE OR REPLACE TABLE gold_catalog.gold_schema.fact_sales AS
SELECT
    li.invoice_line_id,
    li.invoice_id,

    d.date_key,
    dc.customer_key,
    dp.product_key,
    dr.region_key,

    li.quantity,
    li.unit_price,
    ROUND(COALESCE(li.discount, 0), 2) AS discount,

    -- Gross amount
    ROUND(li.quantity * li.unit_price, 2) AS gross_amount,

    -- Discount amount (discount already in decimal format: 0.26 = 26%)
    ROUND(
        li.quantity * li.unit_price * COALESCE(li.discount, 0),
        2
    ) AS discount_amount,

    -- Net amount (after discount)
    ROUND(
        (li.quantity * li.unit_price) * (1 - COALESCE(li.discount, 0)),
        2
    ) AS net_amount,

    inv.currency,
    COALESCE(er.rate_to_usd, 1) AS rate_to_usd,

    -- USD revenue
    ROUND(
        ((li.quantity * li.unit_price) * (1 - COALESCE(li.discount, 0))) * COALESCE(er.rate_to_usd, 1),
        2
    ) AS revenue_usd,

    inv.invoice_status

FROM silver_catalog.silver_schema.invoice_line_items li
JOIN silver_catalog.silver_schema.invoices inv
    ON li.invoice_id = inv.invoice_id

LEFT JOIN silver_catalog.silver_schema.exchange_rates er
    ON inv.currency = er.currency
   AND inv.invoice_date = er.date

JOIN gold_catalog.gold_schema.dim_customer dc
    ON inv.customer_id = dc.customer_id

JOIN gold_catalog.gold_schema.dim_product dp
    ON li.product_id = dp.product_id

JOIN gold_catalog.gold_schema.dim_region dr
    ON inv.region = dr.region_name

JOIN gold_catalog.gold_schema.dim_date d
    ON inv.invoice_date = d.full_date;